In [1]:
import os
import json
import time
import logging
import subprocess
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(dotenv_path="../.env")

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [2]:
# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")

[2026-03-01 18:31:57,260] p22886 {21338686.py:3} INFO - GROQ_API_KEY is set
[2026-03-01 18:31:57,261] p22886 {21338686.py:13} INFO - COHERE_API_KEY is set (for re-ranking)


In [3]:
from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
# You can change this to other models like:
# - "groq/llama-3.3-70b-versatile" (larger, FREE)
# - "gpt-4o-mini" (OpenAI, paid)
# - "claude-3-5-haiku-20241022" (Anthropic, paid)

MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/s7/7ttf9mj16ms92j4k34q3lghr0000gn/T/ipykernel_22886/4098160048.py:11: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  llm = ChatLiteLLM(
[2026-03-01 18:32:04,185] p22886 {4098160048.py:15} INFO - Using model: groq/llama-3.1-8b-instant


In [ ]:
CODEBASE_PATH = "../mcp-gateway-registry"

test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository?",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system?",
]

# Build Bash Tool

In [5]:
def run_bash(command: str, cwd: str = CODEBASE_PATH, max_chars: int = 8000) -> str:
    """
    Execute a bash command inside the codebase directory and return the output.

    Args:
        command:   The shell command to run
        cwd:       Working directory (defaults to codebase root)
        max_chars: Truncate output if it exceeds this length to avoid
                   overwhelming the LLM context window

    Returns:
        Command stdout as a string, truncated if too long
    """
    logger.info(f"Running bash: {command}")
    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=cwd,
            capture_output=True,
            text=True,
            timeout=30
        )
        output = result.stdout
        # Fall back to stderr if stdout is empty
        if not output and result.stderr:
            output = result.stderr
        # Truncate if too large to avoid overwhelming the LLM
        if len(output) > max_chars:
            output = output[:max_chars] + f"\n... [truncated - {len(output)} total chars]"
        return output.strip()
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds."
    except Exception as e:
        return f"Error running command: {e}"


# Quick test - verify the bash tool works
test_output = run_bash("echo 'Bash tool working!' && pwd")
logger.info(test_output)

[2026-03-01 18:32:04,193] p22886 {1157452684.py:14} INFO - Running bash: echo 'Bash tool working!' && pwd
[2026-03-01 18:32:04,207] p22886 {1157452684.py:40} INFO - Bash tool working!
/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/mcp-gateway-registry


# Build Query Classifier

In [ ]:
def classify_query(question: str) -> str:
    """
    Classify a user question to determine which bash retrieval strategy to use.

    Args:
        question: The user's natural language question about the codebase

    Returns:
        One of: 'dependency', 'structure', 'auth', 'api', 'docs', 'code_search'
    """
    q = question.lower()

    # Dependency / package questions
    if any(kw in q for kw in [
        "depend", "package", "requirement", "library", "install",
        "pyproject", "package.json", "pip", "npm"
    ]):
        return "dependency"

    # Structure / layout / language questions
    if any(kw in q for kw in [
        "entry point", "main file", "structure", "language", "file type",
        "directory", "folder", "layout", "programming language", "what files"
    ]):
        return "structure"

    # Authentication / authorization questions
    if any(kw in q for kw in [
        "auth", "token", "oauth", "jwt", "login", "authorization",
        "scope", "credential", "keycloak", "cognito", "permission", "validate"
    ]):
        return "auth"

    # API endpoint questions
    if any(kw in q for kw in [
        "endpoint", "api", "route", "rest", "http", "request", "response"
    ]):
        return "api"

    # How-to / architecture / modification questions
    if any(kw in q for kw in [
        "how", "add", "implement", "support", "modify", "interface",
        "architecture", "would you", "what files would"
    ]):
        return "docs"

    # Default: generic code search
    return "code_search"


for q in test_questions:
    logger.info(f"[{classify_query(q)}] {q}")

[2026-03-01 18:32:04,259] p22886 {2824859674.py:62} INFO - [dependency] What Python dependencies does this project use?
[2026-03-01 18:32:04,259] p22886 {2824859674.py:62} INFO - [structure] What is the main entry point file for the registry service?
[2026-03-01 18:32:04,260] p22886 {2824859674.py:62} INFO - [structure] What programming languages and file types are used in this repository?
[2026-03-01 18:32:04,260] p22886 {2824859674.py:62} INFO - [auth] How does the authentication flow work, from token validation to user authorization?
[2026-03-01 18:32:04,260] p22886 {2824859674.py:62} INFO - [auth] What are all the API endpoints available in the registry service and what scopes do they require?
[2026-03-01 18:32:04,260] p22886 {2824859674.py:62} INFO - [auth] How would you add support for a new OAuth provider (e.g., Okta) to the authentication system?


# Build Context Retriever

In [14]:
def retrieve_context(question: str, query_type: str) -> str:
    """
    Select and execute bash commands appropriate for the query type.
    Context is kept under 4000 chars to avoid Groq TPM rate limits.

    Args:
        question:   The original user question
        query_type: Category returned by classify_query()

    Returns:
        Retrieved context as a single string to pass to the LLM
    """
    parts = []

    if query_type == "dependency":
        # Only read root pyproject.toml, grep dependencies section only
        parts.append("=== pyproject.toml - dependencies ===")
        parts.append(run_bash(
            "grep -A 60 '\\[project\\]' pyproject.toml | head -70", max_chars=2500
        ))

        parts.append("\n=== auth_server/pyproject.toml - dependencies ===")
        parts.append(run_bash(
            "grep -A 30 'dependencies' auth_server/pyproject.toml | head -35", max_chars=1000
        ))

        parts.append("\n=== cli/package.json (name + dependencies only) ===")
        parts.append(run_bash(
            "python3 -c \"import json; d=json.load(open('cli/package.json')); "
            "print('name:', d.get('name')); "
            "[print(k+': '+v) for k,v in d.get('dependencies',{}).items()]\"",
            max_chars=800
        ))

    elif query_type == "structure":
        # Only tree + file types + first 40 lines of main.py + Dockerfile CMD
        parts.append("=== Directory Structure (depth 2) ===")
        parts.append(run_bash("tree -L 2 --dirsfirst", max_chars=1500))

        parts.append("\n=== File Types Count ===")
        parts.append(run_bash(
            "find . -not -path './.git/*' -type f | "
            "sed 's/.*\\.//' | sort | uniq -c | sort -rn | head -15"
        ))

        parts.append("\n=== registry/main.py (first 40 lines) ===")
        parts.append(run_bash("head -40 registry/main.py"))

        parts.append("\n=== Entry point from Dockerfile ===")
        parts.append(run_bash("grep -E 'CMD|ENTRYPOINT|python|uvicorn' Dockerfile"))

    elif query_type == "auth":
        # Focused grep instead of reading full files
        parts.append("=== auth_server/server.py (first 80 lines) ===")
        parts.append(run_bash("head -80 auth_server/server.py", max_chars=2000))

        parts.append("\n=== OAuth providers config ===")
        parts.append(run_bash("cat auth_server/oauth2_providers.yml", max_chars=1000))

        parts.append("\n=== Auth function definitions ===")
        parts.append(run_bash(
            r"grep -rn 'def.*token\|def.*auth\|def.*scope\|def.*validate' "
            "--include='*.py' registry/auth/ | head -30"
        ))

        parts.append("\n=== docs/auth.md (first 60 lines) ===")
        parts.append(run_bash("head -60 docs/auth.md", max_chars=1500))

    elif query_type == "api":
        # Route decorators + scopes only
        parts.append("=== API route decorators ===")
        parts.append(run_bash(
            r"grep -rn '@router\.' --include='*.py' registry/api/ | head -40"
        ))

        parts.append("\n=== Scope requirements ===")
        parts.append(run_bash(
            r"grep -rn 'require_scope\|security' "
            "--include='*.py' registry/api/ | head -30"
        ))

        parts.append("\n=== auth_server/scopes.yml ===")
        parts.append(run_bash("cat auth_server/scopes.yml", max_chars=1500))

        parts.append("\n=== registry/api/ files ===")
        parts.append(run_bash("find registry/api -name '*.py' | sort"))

    elif query_type == "docs":
        # Provider list + one example + architecture overview only
        parts.append("=== auth_server/providers/ files ===")
        parts.append(run_bash("ls auth_server/providers/"))

        parts.append("\n=== Example provider implementation (first 60 lines) ===")
        parts.append(run_bash(
            "head -60 $(ls auth_server/providers/*.py | head -1)", max_chars=1500
        ))

        parts.append("\n=== oauth2_providers.yml ===")
        parts.append(run_bash("cat auth_server/oauth2_providers.yml", max_chars=1000))

        parts.append("\n=== docs/auth.md (first 50 lines) ===")
        parts.append(run_bash("head -50 docs/auth.md", max_chars=1000))

        parts.append("\n=== docs/registry-auth-architecture.md (first 50 lines) ===")
        parts.append(run_bash("head -50 docs/registry-auth-architecture.md", max_chars=1000))

    else:  # code_search
        parts.append("=== registry/ structure ===")
        parts.append(run_bash("tree -L 2 --dirsfirst registry/", max_chars=1000))
        parts.append(run_bash(
            f"grep -rn '{question[:30]}' --include='*.py' -l | head -10"
        ))

    context = "\n".join(parts)
    logger.info(f"Retrieved context length: {len(context)} chars")
    return context


logger.info("retrieve_context() defined")

[2026-03-01 18:41:39,092] p22886 {277978338.py:119} INFO - retrieve_context() defined


# Retrieval and Generation

In [15]:
from langchain_core.prompts import ChatPromptTemplate

# System prompt for code Q&A - instructs the model to cite file names
system_prompt = (
    "You are an expert software engineer analyzing the mcp-gateway-registry codebase. "
    "You are given context retrieved from the codebase using bash tools (grep, find, cat, tree). "
    "Answer the user's question based ONLY on the provided context. "
    "Always cite specific file names when referencing code. "
    "If the context is insufficient to answer fully, say so clearly. "
    "Keep your answer concise but thorough.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

logger.info("Prompt template configured")

[2026-03-01 18:41:39,098] p22886 {667597815.py:21} INFO - Prompt template configured


In [16]:
def answer_question(question: str) -> dict:
    """
    Full pipeline: classify → retrieve context via bash → generate answer with LLM.

    Args:
        question: The user's question about the codebase

    Returns:
        dict with keys: 'question', 'query_type', 'context', 'answer'
    """
    logger.info(f"Question: {question}")

    # Step 1: Classify the query
    query_type = classify_query(question)
    logger.info(f"Query type: {query_type}")

    # Step 2: Retrieve context using bash tools
    context = retrieve_context(question, query_type)

    # Step 3: Build the prompt and invoke the LLM
    messages = prompt.format_messages(context=context, input=question)
    for attempt in range(5):
        try:
            response = llm.invoke(messages)
            answer = response.content
            logger.info(f"Answer (preview): {answer[:300]}")
            return {
                "question": question,
                "query_type": query_type,
                "context": context,
                "answer": answer,
            }
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 30 * (attempt + 1)
                logger.warning(f"Rate limit hit, waiting {wait}s... (attempt {attempt+1}/5)")
                time.sleep(wait)
            else:
                raise e
    
    raise RuntimeError("Failed after 5 retries")

logger.info("answer_question() with retry defined")

[2026-03-01 18:41:39,104] p22886 {1131534857.py:43} INFO - answer_question() with retry defined


In [17]:
# Quick smoke test with the simplest question
test_result = answer_question("What Python dependencies does this project use?")
logger.info(f"Smoke test answer:\n{test_result['answer']}")

[2026-03-01 18:41:39,108] p22886 {1131534857.py:11} INFO - Question: What Python dependencies does this project use?
[2026-03-01 18:41:39,108] p22886 {1131534857.py:15} INFO - Query type: dependency
[2026-03-01 18:41:39,108] p22886 {1157452684.py:14} INFO - Running bash: grep -A 60 '\[project\]' pyproject.toml | head -70
[2026-03-01 18:41:39,125] p22886 {1157452684.py:14} INFO - Running bash: grep -A 30 'dependencies' auth_server/pyproject.toml | head -35
[2026-03-01 18:41:39,144] p22886 {1157452684.py:14} INFO - Running bash: python3 -c "import json; d=json.load(open('cli/package.json')); print('name:', d.get('name')); [print(k+': '+v) for k,v in d.get('dependencies',{}).items()]"
[2026-03-01 18:41:39,192] p22886 {277978338.py:115} INFO - Retrieved context length: 2630 chars
18:41:39 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-01 18:41:39,194] p22886 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant

## Answer Test Questions

In [ ]:
all_results = []

for i, question in enumerate(test_questions, 1):
    logger.info(f"{'='*60}")
    logger.info(f"Question {i}/{len(test_questions)}: {question}")
    result = answer_question(question)
    all_results.append(result)
    logger.info(f"Answer {i}:\n{result['answer']}")
    time.sleep(10)

[2026-03-01 18:41:39,854] p22886 {4192524146.py:13} INFO - ============================================================
[2026-03-01 18:41:39,855] p22886 {4192524146.py:14} INFO - Question 1/6: What Python dependencies does this project use?
[2026-03-01 18:41:39,855] p22886 {1131534857.py:11} INFO - Question: What Python dependencies does this project use?
[2026-03-01 18:41:39,856] p22886 {1131534857.py:15} INFO - Query type: dependency
[2026-03-01 18:41:39,856] p22886 {1157452684.py:14} INFO - Running bash: grep -A 60 '\[project\]' pyproject.toml | head -70
[2026-03-01 18:41:39,875] p22886 {1157452684.py:14} INFO - Running bash: grep -A 30 'dependencies' auth_server/pyproject.toml | head -35
[2026-03-01 18:41:39,890] p22886 {1157452684.py:14} INFO - Running bash: python3 -c "import json; d=json.load(open('cli/package.json')); print('name:', d.get('name')); [print(k+': '+v) for k,v in d.get('dependencies',{}).items()]"
[2026-03-01 18:41:39,916] p22886 {277978338.py:115} INFO - Retrieved

# Save Answers

In [19]:
# Save all results to part1_results.txt
with open("part1_results.txt", "w") as f:
    f.write("Part 1: Code Q&A System with Bash Tools - Results\n")
    f.write("=" * 60 + "\n\n")

    for i, result in enumerate(all_results, 1):
        f.write(f"Question {i} [{result['query_type']}]:\n")
        f.write(f"{result['question']}\n")
        f.write("-" * 40 + "\n")
        f.write(f"{result['answer']}\n\n")
        f.write("=" * 60 + "\n\n")

logger.info("Saved to part1_results.txt")

[2026-03-01 18:42:45,257] p22886 {3576768567.py:13} INFO - Saved to part1_results.txt
